In [150]:
import jax 
import jax.numpy as jnp 
import numpy as np 

seed = 42 
key = jax.random.PRNGKey(seed=seed)

class key_gen:
    def __init__(self, seed=42):
        self.seed = seed 
        self.key = jax.random.PRNGKey(seed)
        
    def get_key(self):
        self.key, return_key = jax.random.split(self.key)
        return return_key
        
kg = key_gen(seed)

In [151]:
elements = jnp.array([10, 20, 404, 123, 512, 908])
element_to_index = {
    x.item(): i 

    for i, x in enumerate(elements)
}
p_values = jnp.array([0.1, 0.25, 0.2, 0.05, 0.15, 0.25])
(elements, p_values, element_to_index)

(Array([ 10,  20, 404, 123, 512, 908], dtype=int32),
 Array([0.1 , 0.25, 0.2 , 0.05, 0.15, 0.25], dtype=float32),
 {10: 0, 20: 1, 404: 2, 123: 3, 512: 4, 908: 5})

In [152]:
elements_2 = np.array(elements.tolist())
print(type(elements_2[0].item()))

<class 'int'>


In [153]:
for i, element in enumerate(elements):
    print(f"index: {type(i)}, Element: {type(element.item())}")

index: <class 'int'>, Element: <class 'int'>
index: <class 'int'>, Element: <class 'int'>
index: <class 'int'>, Element: <class 'int'>
index: <class 'int'>, Element: <class 'int'>
index: <class 'int'>, Element: <class 'int'>
index: <class 'int'>, Element: <class 'int'>


In [154]:
rng = np.random.default_rng()
rng.choice(elements, p=p_values)

np.int32(908)

In [155]:
np.random.normal(0, 0.2)

0.04728713936457829

In [156]:
count = np.zeros(shape=elements.shape)
for x in range(10_000):
    choice = rng.choice(elements, size=1, p=p_values).item()
    index = element_to_index[choice]
    count[index] += 1

selection_probability = count/count.sum()
print(f" Actual P_values: {p_values} \n selected_prob: {selection_probability}")
print(f"sum of selection probabilities: {selection_probability.sum().item()}")

 Actual P_values: [0.1  0.25 0.2  0.05 0.15 0.25] 
 selected_prob: [0.0945 0.2485 0.205  0.0523 0.148  0.2517]
sum of selection probabilities: 1.0


In [157]:
class Environment:
    """
        # all the actions considered inside are one based indexing
    """

    def __init__(
            self,
            n_actions: int,
            context_size: int,
            seed = 42
        ):
        self.n_actions = n_actions
        self.context_size = context_size 
        self.rng = np.random.default_rng(seed)
    
    def sample_context(self):
        return self.rng.integers(low=-1, size=2, high=2)

    def encode_context(self, context: np.array):
        x1, x2 = context 
        return (f"| x1:{x1} x2:{x2}")

    def get_reward_array(self, context):
        x1, x2 = context 
        return np.array([
            (-2*x1 -x2),
            (-x1 + 10*x2), 
            (5*x1 - 4*x2), 
            (x1 + 3*x2)            
        ])

    def reward(self, context: tuple, action: int) -> int:
        return self.get_reward_array(context)[action-1] + np.random.normal(0, 0.2)

    def optimal_action(self, context):
        action_0 = np.argmax(self.get_reward_array(context))
        return action_0 + 1

    def encode_learn_example(self, context, action, reward, action_probs):
        cost = -reward 
        action_probability = action_probs[action-1]
        x1, x2 = context 
        return (
            f"{action}:{cost:.2f}:{action_probability} | x1:{x1} x2:{x2}"
            )


In [158]:
import vowpalwabbit 

env = Environment(n_actions=4, context_size=2)
vw = vowpalwabbit.Workspace(
    "--cb_explore 4 --epsilon 0.15",
    quiet=True
)


In [159]:
actions = [
    
]

In [160]:
for i in range(10_000):
    context = env.sample_context()
    encoded_context = env.encode_context(context)
    action_probs = vw.predict(encoded_context)
    # print(f"action_probs at step: {i} = {action_probs}")
    
    # action = rng.choice(np.arange(1, 5), p=action_probs) 
    action = np.argmax(action_probs) + 1

    reward = env.reward(context, action)
    optimal_action = env.optimal_action(context)
    vw.learn(env.encode_learn_example(context, action, reward, action_probs))

    actions.append({
        "i": i, 
        "context": str(context), 
        "action": action,
        "reward": reward,
        "optimal_action": optimal_action, 
    })

In [161]:
import pandas as pd
df = pd.DataFrame(actions)

In [162]:
df[df["action"] == df["optimal_action"]].describe()

,i,action,reward,optimal_action
count,7975.000000,7975.000000,7975.000000,7975.000000
mean,5067.362759,2.235110,7.072150,2.235110
std,2900.921611,0.732129,3.077453,0.732129
min,1.000000,1.000000,-0.359826,1.000000
25%,2551.000000,2.000000,4.014818,2.000000
50%,5103.000000,2.000000,8.814600,2.000000
75%,7598.500000,3.000000,9.811981,3.000000
max,9998.000000,3.000000,11.706489,3.000000
